# Suite EXP — Expressions

An expression is an `Expressions.Framework.Expression`, a reference object whose `root` is a term: a value of
`Expressions.Framework.Term`, a union of the kinds `literal`, `call`, `variable` and `let` and the container
constructors `list` and `mapping`, to which Basic adds `quantifier` by schema extension. A term carries no domain.
`Framework.install` and `Basic.install` define them in a store; terms are built there,
in a transaction, through mbse-schemas' builders (or Basic's `Writer`s, which write through them), and read through its
readers, so they serialize, validate and compare like any other value. `Evaluators` computes them (suite EVL).

In [ ]:
import { Evaluators as V, Expressions as E, Framework } from "@mbse/expressions";
import * as Basic from "@mbse/expressions/Dialects/Basic";
import { Terms as F } from "@mbse/expressions/Framework";
import { Builders, Comparison as C, Errors, JSON, Plain, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, equal, form_equal, map, raises, same } from "./support.js";

const { ValueError } = Errors;

const store: any = new Builders.Store();
store.transaction(Framework.install);
store.transaction(Basic.install);
const Term: any = store.registered("Expressions.Framework.Term");
const Expression: any = store.registered("Expressions.Framework.Expression");

/** The term value `spec` writes, built in a transaction. */
function term(spec: unknown): any {
  return store.transaction(() => store.Expressions.Framework.Term(E.DIALECT.spec(spec)));
}

const lit = term(E.literal(3n));
const op = term(E.operation("f", 1n, "x", lit));

## EXP-01 · Literals hold one basic scalar

In [ ]:
assert(F.kindOf(lit)[0] === "literal" && form_equal(F.form(lit), new F.Form("literal", { value: 3n })) && E.DIALECT.validate(lit).length === 0);
assert(lit.literal.value.get() === 3n && lit.as_schema() === Term); // read through the readers
for (const value of [0n, 2n ** 70n, -0, NaN, "", new Uint8Array(), false]) { // every basic scalar
  assert(E.DIALECT.validate(term(value)).length === 0 && F.same(term(value), term(E.literal(value))));
}
assert(typeof F.form(term(-0)).attributes.get("value") === "number" && Object.is(term(-0).literal.value.get(), -0));
assert(same(E.DIALECT.validate(term(E.literal(null))), ["a literal needs a value"]));
assert(same(E.DIALECT.validate(term((u: any) => u.literal((r: any) => r))), ["a literal needs a value"]));
assert(F.same(term((u: any) => u.literal((r: any) => r.value((p: any) => p.as_scalar("x")))), term("x"))); // the builders' own calls
// value is an OfAny of the five basic scalars: its schema refuses anything else
raises(TypeError, () => term((u: any) => u.literal((r: any) => r.value(lit))),
  "property 'value' does not hold a value of Expressions.Framework.Term");
raises(TypeError, () => term((u: any) => u.literal((r: any) => r.value(op.call.arguments))),
  "property 'value' does not hold a value of a container");

## EXP-02 · Calls have a callable and ordered arguments

In [ ]:
let form = F.form(op);
assert(form.kind === "call" && equal(form.attributes, map({ callable: "f" })) && same(form.arguments.map((a: any) => a.literal.value.get()), [1n, "x", 3n]));
assert(F.same(form.arguments[2], lit) && op.call.callable.get() === "f");
assert(same(E.DIALECT.validate(op), ["'f' is not an operation Basic defines"])); // written, and refused: Basic defines no f
const nested = term((u: any) => u.call((c: any) => c.callable("not").arguments([E.DIALECT.spec(op)])));
assert(F.same(F.form(nested).arguments[0], op));
assert(same(E.DIALECT.validate(term((u: any) => u.call((c: any) => c))), ["a call needs a callable"]));
assert(same(E.DIALECT.validate(term((u: any) => u.call((c: any) => c.callable("")))), ["a call needs a callable"]));
assert(same(E.DIALECT.validate(term((u: any) => u.call((c: any) => c.callable(3n)))), ["a call's callable must be a str, got int"]));
const bad = term(E.operation("f", E.literal(null), (u: any) => u.call((c: any) => c.arguments([E.literal(null)]))));
assert(same(E.DIALECT.validate(bad), [
  "'f' is not an operation Basic defines",
  "argument 0: a literal needs a value",
  "argument 1: a call needs a callable",
  "argument 1: argument 0: a literal needs a value",
]));

## EXP-03 · A term is a nested value: written into another it is copied, so it holds no cycle

In [ ]:
const pair = term(E.operation("g", op, op)); // each argument a copy of op
assert(F.form(pair).arguments.every((a) => F.same(a, op)));
assert([...F.walk(pair)].length === 1 + 2 * [...F.walk(op)].length); // nothing is shared: every term is visited
let again = term(pair);
assert(F.same(again, pair) && again !== pair);

## EXP-04 · Terms are built through mbse-schemas' builders, in a transaction: a term alone, or an expression's root

In [ ]:
const x = E.variable("x");
const source = store.transaction(() => store.Expressions.Framework.Expression((e: any) => e.root(E.operation("f", 1n))));
assert(Expression.ref && equal(F.form(F.term(source)).attributes, map({ callable: "f" })));
const [updated, copy] = store.transaction(() => [
  store.Expressions.Framework.Expression(source, (e: any) => e.root(E.operation("g", 2n))), // an update
  store.Expressions.Framework.Expression(source), // a clone
]);
assert(updated === source && equal(F.form(F.term(source)).attributes, map({ callable: "g" })));
assert(copy !== source && F.same(copy, source));
raises(ValueError, () => store.transaction(() => F.term(store.Expressions.Framework.Expression((e: any) => e))), "no term is set");
assert(F.same(F.term(source.root), source)); // an expression, its root property, or the term itself
raises(TypeError, () => F.term(3n), "not a term: 3");
// A spec is a writer, a term value (copied), a native value (a literal) or a callable on the term value's builder.
assert(E.DIALECT.spec(x) instanceof E.Writer && E.DIALECT.spec(lit) instanceof F.Writer);
assert(E.DIALECT.spec(1n) instanceof E.Writer && E.DIALECT.spec((u: any) => u.variable((v: any) => v.name("x"))) instanceof F.Writer);
raises(TypeError, () => E.DIALECT.spec(BigInt), "expected a term, a writer or a native value, got <class 'int'>");
raises(TypeError, () => E.DIALECT.spec([1n]), "expected a term, a writer or a native value, got [1]");
raises(TypeError, () => F.spec(null), "expected a term, a writer or a native value, got None");
raises(TypeError, () => F.spec(1n, () => null), "expected a term, a writer or a native value, got 1"); // a dialect with no literal for it

## EXP-05 · The framework's schemas are an install: ordinary, registered schemas, which Basic's install extends

In [ ]:
const before = JSON.ToJSON(store).Snapshot();
store.transaction(Framework.install);
store.transaction(Basic.install);
assert(JSON.ToJSON(store).Snapshot() === before); // run again, each install writes nothing
let names = ["Expressions.Framework.Literal", "Expressions.Framework.Call", "Expressions.Framework.Variable",
  "Expressions.Framework.Let", "Expressions.Framework.List", "Expressions.Framework.Mapping", "Expressions.Basic.Quantifier"];
const kinds = names.map((name) => store.registered(name));
const Pair: any = store.registered("Expressions.Framework.Pair");
for (const schema of [Term, Expression, store.registered("Expressions.Framework.HasRoot"), Pair, ...kinds]) {
  assert(schema.validate().length === 0);
}
assert(same(Term.branches.map((b: any) => b.name), ["literal", "call", "variable", "let", "list", "mapping", "quantifier"]));
assert(same(Term.branches.map((b: any) => b.schema), kinds) && !kinds.some((kind) => kind.ref)); // value objects
assert(equal(kinds.map((k) => [...k.properties.keys()]), [["value"], ["callable", "arguments"], ["name"], ["name", "value", "body"],
  ["elements"], ["entries"], ["name", "quantifier", "collection", "body"]]));
assert(kinds[4].properties.get("elements").schema.element === Term && kinds[5].properties.get("entries").schema.element === Pair);
assert(same([...Pair.properties.keys()], ["key", "value"]) && [...Pair.properties.values()].every((p: any) => p.schema === Term));
assert(kinds[1].properties.get("arguments").schema.element === Term && kinds[3].properties.get("body").schema === Term);
assert(Expression.ref && same([...Expression.properties.keys()], ["root"]) && Expression.properties.get("root").schema === Term);
const Dialect: any = store.registered("Expressions.Framework.Dialect"); // a kind union: each dialect adds its singleton's schema
assert(same(Dialect.branches.map((b: any) => b.name), ["basic"]) && Dialect.branches[0].schema === store.registered("Expressions.Basic.Dialect"));
assert(Dialect.branches[0].schema.ref && Dialect.branches[0].schema.singleton === "Expressions.Basic");
const HasRoot: any = store.registered("Expressions.Framework.HasRoot");
assert(same([...HasRoot.entry.links.keys()], ["schema", "expression"]) && HasRoot.uniques.length === 1
  && same([...HasRoot.uniques[0]], ["expression"]));
const OfExpression: any = store.registered("Schemas.OfExpression");
assert(OfExpression.adjacencies.get("expression").relation === HasRoot && OfExpression.adjacencies.get("expression").me === "schema");

## EXP-06 · Round trips through Plain, JSON and YAML: a term written nested, its kind its branch's name

In [ ]:
const expr = term(E.list_(E.operation("eq", 1n, 1.5), "s", new Uint8Array([1]), E.mapping([true, 1n], ["k", E.variable("v")])));
const expected = JSON.ToJSON(store)(Term, expr);
for (const [encode, decode] of [[Plain.ToPlain(store), Plain.FromPlain(store)], [JSON.ToJSON(store), JSON.FromJSON(store)],
  [YAML.ToYAML(store), YAML.FromYAML(store)]] as [any, any][]) {
  const back = decode(Term, encode(Term, expr));
  assert(back !== expr && F.same(back, expr) && E.DIALECT.validate(back, ["v"]).length === 0);
  assert(JSON.ToJSON(store)(Term, back) === expected);
}
assert(JSON.ToJSON(store)(Term, term(E.operation("not", true)))
  === '{"call": {"callable": "not", "arguments": [{"literal": {"value": true}}]}}');
const alone = JSON.ToJSON(store)(Term, term(-0));
assert(alone === '{"literal": {"value": -0.0}}');
assert(Object.is((JSON.FromJSON(store)(Term, alone) as any).literal.value.get(), -0));
// A literal's value is bare where only one basic scalar writes its form, otherwise wrapped, kind then schema.
for (const [value, wire] of [[1n, "1"], [1.5, "1.5"], [true, "true"], ["x", '{"scalar": {"str": "x"}}'],
  [new Uint8Array([1]), '{"scalar": {"bytes": "AQ=="}}'], [NaN, '{"scalar": {"float": "NaN"}}']] as [unknown, string][]) {
  const written = JSON.ToJSON(store)(Term, term(value));
  assert(written === `{"literal": {"value": ${wire}}}` && F.same(JSON.FromJSON(store)(Term, written), term(value)));
}
// nothing set: only the branch is written
assert(same(["literal", "call", "variable", "let", "list", "mapping"].map((k) => JSON.ToJSON(store)(Term, term((u: any) => u[k]((r: any) => r)))), [
  '{"literal": {}}', '{"call": {}}', '{"variable": {}}', '{"let": {}}', '{"list": {}}', '{"mapping": {}}']));
assert(JSON.ToJSON(store)(Term, term(E.mapping(["a", 1n])))
  === '{"mapping": {"entries": [{"key": {"literal": {"value": {"scalar": {"str": "a"}}}}, "value": {"literal": {"value": 1}}}]}}');
raises(Error, () => JSON.FromJSON(store)(Term, '{"operation": {}}'), "has no branch 'operation'");
// A store holding expressions and Basic's dialect, its domains (a packed one among them) and operations, round-trips
// whole, read into a store with the same installs: nothing lost, nothing written twice.
const defined = store.transaction(Basic.define)[0];
store.transaction(() => {
  store.Expressions.Framework.Expression((e: any) => e.root(expr));
  const state = store.Expressions.Framework.Domain((d: any) => d.kind((k: any) => k.as_value((a: any) => a.as_union(
    (u: any) => u.enum((e: any) => e.members(["IDLE", "RUN"]))))));
  const mode = store.Expressions.Framework.Domain((d: any) => d.kind((k: any) => k.as_value((a: any) => a.as_union((u: any) => u.packed(
    (e: any) => e.codes([0n, 1n]))))).packs((a: any) => a.insert((e: any) => e.domain(state).representation(defined.get("uint8")))));
  store.Expressions.Basic.Dialect(defined.get("dialect"), (b: any) => b.domains({ state: (e: any) => e.domain(state),
    mode: (e: any) => e.domain(mode) }));
});
const snapshot = JSON.ToJSON(store).Snapshot();
const restored: any = new Builders.Store();
restored.transaction(Framework.install);
restored.transaction(Basic.install);
JSON.FromJSON(restored).Snapshot(snapshot);
assert(JSON.ToJSON(restored).Snapshot() === snapshot);
const restoredBasic = restored.singleton("Expressions.Basic");
assert(same([...restoredBasic.domains].map((e: any) => `${e.name}`).sort(), [...Basic.Domains.DOMAINS.keys(), "state", "mode"].sort()));
assert([...restoredBasic.operations].length === E.OPERATIONS.size);
const packed = [...restoredBasic.domains].find((e: any) => `${e.name}` === "mode").domain;
assert(Basic.Domains.read(packed).name() === "packed(enum(IDLE, RUN) as uint8)" && Basic.Domains.read(packed).validate().length === 0);

## EXP-07 · An expression is a reference object; a schema's expression is linked by `HasRoot`, an extension adjacency

In [ ]:
const [adult, schema] = store.transaction(() => {
  const adult = store.Expressions.Framework.Expression((e: any) => e.root(E.variable("person").age.ge(18n)));
  const schema = store.Schemas.OfExpression((s: any) => s.name("exp.Adult").result([store.Schemas.OfBool])
    .expression((a: any) => a.insert((e: any) => e.expression(adult))));
  return [adult, schema];
});
const linked = [...(new Builders.OfExpression.Builder(schema, false, store) as any).expression].map((entry: any) => entry.expression);
assert(linked.length === 1 && F.same(linked[0], adult));
assert(Validators.Validate(store)(Expression, adult).length === 0);
assert(V.OfAny(adult, { person: null }) === null);

## EXP-08 · Validation and comparison work through the schemas

In [ ]:
assert(Validators.Validate(store)(Expression, adult).length === 0 && E.DIALECT.validate(adult, ["person"]).length === 0);

function compare(a: unknown, b: unknown) {
  return new C.OfUnion(Term, term(a)).compare(new C.OfUnion(Term, term(b)));
}

assert(compare(1n, 1n) === 0 && compare(1n, 2n) === null); // values are not ordered
assert(compare(1n, 1.0) === null); // another scalar
assert(compare(E.operation("f", 1n), E.operation("f", 1n)) === 0 && compare(E.operation("f", 1n), E.operation("f", 2n)) === null);
assert(compare(E.variable("x"), E.literal("x")) === null); // another branch

## EXP-09 · Terms read and write through the visitor protocols

In [ ]:
names = [...op.call.as_properties].map((p: any) => p.as_schema().name);
assert(same(names, ["callable", "arguments"])); // the properties set, schema unaware
const entries: [unknown, any][] = [];
op.call.arguments.as_value((a: any) => a.as_container((c: any) => c.as_entries((it: Iterator<[unknown, any]>) => {
  for (let next = it.next(); !next.done; next = it.next()) entries.push(next.value);
})));
assert(same(entries.map(([i]) => i), [0n, 1n, 2n]));
const values: unknown[] = [];
for (const [, element] of entries) {
  element.as_union((u: any) => values.push(u.literal.value.get()));
}
assert(same(values, [1n, "x", 3n]) && F.same(F.term(entries[0]![1]), term(1n)) && !F.isTerm(entries[0]![1])); // an element's reader holds a term
const seen: string[] = [];
term((u: any) => {
  seen.push(u.as_schema().constructor.name);
  return u.literal((r: any) => {
    seen.push(r.as_schema().name);
    return r;
  });
});
assert(same(seen, ["UnionData", "Expressions.Framework.Literal"]));
// A writer writes through what it is given: a term value's union, a property, a container's element.
const written = store.transaction(() => store.Expressions.Framework.Expression((e: any) => e.root((p: any) => p.as_value((a: any) => a.as_union(
  (u: any) => u.call((c: any) => c.callable("f").arguments([E.literal(1n), (y: any) => y.as_union(E.variable("x").write)])))))));
assert(F.same(written, term(E.operation("f", 1n, E.variable("x")))));

## EXP-10 · Variables and lets

In [ ]:
const v = term(E.variable("x"));
assert(F.nameOf(v) === "x" && F.kindOf(v)[0] === "variable");
assert(same(E.DIALECT.validate(v), ["variable 'x' is not bound"]) && E.DIALECT.validate(v, ["x"]).length === 0);
assert(same(E.DIALECT.validate(term((u: any) => u.variable((r: any) => r))), ["a variable needs a name"]));
assert(same(E.DIALECT.validate(term((u: any) => u.variable((r: any) => r.name(1n)))), ["a variable's name must be a str, got int"]));
const let_ = term(E.let_("x", 1n, E.variable("x")));
form = F.form(let_);
assert(equal(form.attributes, map({ name: "x" })) && F.same(form.arguments[0], term(1n)) && F.same(form.arguments[1], v));
assert(E.DIALECT.validate(let_).length === 0 && F.nameOf(let_) === "x");
assert(same(E.DIALECT.validate(term(E.let_("x", null, null))), ["a let needs a value", "a let needs a body"]));
assert(same(E.DIALECT.validate(term((u: any) => u.let((r: any) => r.value(E.literal(1n)).body(E.variable("x"))))), [
  "a let needs a name", "body: variable 'x' is not bound"]));
const sameName = term(E.let_("y", E.variable("y"), E.variable("y")));
assert(same(E.DIALECT.validate(sameName), ["value: variable 'y' is not bound"])); // the name is bound in the body only
const openLet = term(E.let_("x", null, null));
assert(same(F.form(openLet).arguments, [null, null]) && F.nameOf(term(E.literal(1n))) === null);
assert(F.same(term(E.DIALECT.make(F.form(openLet))), openLet));

## EXP-11 · Writers build terms with methods

In [ ]:
const person = E.variable("person");
const writer = person.age.ge(18n);
assert(writer instanceof E.Writer && typeof writer === "function");
form = F.form(term(writer));
assert(equal(form.attributes, map({ callable: "ge" })) && (form.arguments[1] as any).literal.value.get() === 18n);
const got = F.form(form.arguments[0]);
assert(equal(got.attributes, map({ callable: "get" })) && F.nameOf(got.arguments[0]) === "person"
  && (got.arguments[1] as any).literal.value.get() === "age");
assert((F.form(term(person.get("eq"))).arguments[1] as any).literal.value.get() === "eq"); // a property named like a method
const methods = [person.eq(1n), person.ne(1n), person.lt(1n), person.le(1n), person.gt(1n), person.ge(1n), person.and_(true), person.or_(true),
  person.not_(), person.implies(true), person.add(1n), person.sub(1n), person.mul(1n), person.neg(), person.has("x"),
  person.get("x")];
assert(same(methods.map((m) => F.form(term(m)).attributes.get("callable")), [
  "eq", "ne", "lt", "le", "gt", "ge", "and", "or", "not", "implies", "add", "sub", "mul", "neg", "has", "get"]));
assert(term(E.literal(3n)).literal.value.get() === 3n && equal(F.form(term(E.operation("in", 1n, 2n))).attributes, map({ callable: "in" })));
const bound = term(E.let_("a", 1n, E.variable("a")));
assert(F.nameOf(bound) === "a" && F.same(term(bound), bound));
// JavaScript's own probes are not properties, nor are private names, as Python's are not
assert(person._private === undefined && person.then === undefined && person.toJSON === undefined && person[Symbol.iterator as any] === undefined);

## EXP-13 · Dialects, operations and domains are data: a dialect's singleton names its operations and owns its domains

In [ ]:
const definedAgain = store.transaction(Basic.define)[0]; // Basic's dialect's data; run again, it writes nothing
assert(store.transaction(Basic.define)[0].get("uint8") === definedAgain.get("uint8") && definedAgain.get("uint8") === defined.get("uint8"));
const basic = definedAgain.get("dialect");
assert(basic === store.singleton("Expressions.Basic")); // its identity: its global name
const [Defines, Operates]: any[] = [store.registered("Expressions.Framework.Defines"), store.registered("Expressions.Framework.Operates")];
assert(same([...Defines.entry.links.keys()], ["dialect", "domain"]) && same([...Defines.entry.properties.keys()], ["name"]));
assert(equal(Defines.uniques.map((u: any) => [...u].sort()), [["dialect", "name"], ["domain"]]));
assert(same([...Operates.entry.links.keys()], ["dialect", "operation"]) && same([...Operates.entry.properties.keys()], ["name"]));
assert(equal(Operates.uniques.map((u: any) => [...u].sort()), [["dialect", "name"], ["operation"]]));
const DomainSchema: any = store.registered("Expressions.Framework.Domain");
assert(DomainSchema.ref && DomainSchema.adjacencies.get("dialect").owner && DomainSchema.adjacencies.get("dialect").relation === Defines);
// Its operations: expression schemas its module defines, named by the dialect; their variables, in order, its arguments.
const operations = new Map<string, any>([...basic.operations].map((entry: any) => [`${entry.name}`, entry.operation]));
assert(same([...operations.keys()], [...E.OPERATIONS.keys()]) && same([...E.OPERATIONS.keys()].sort(), [...Basic.Domains.SIGNATURES.keys()].sort()));
assert(equal(new Map([...operations].map(([name, o]) => [name, o.pattern.variables.size])), E.CORE));
assert([...operations].every(([name, o]) => o === store.registered(`Expressions.Basic.${name}`) && o.validate().length === 0));
const [add, number] = [operations.get("add"), store.registered("Expressions.Basic.Number")];
assert(same([...add.pattern.variables.keys()], ["a", "b"]) && add.pattern.variables.get("a").schema === number && add.result === number);
assert(same([...operations.get("shl").pattern.variables.keys()], ["value", "count"])); // the order declared: arguments are positional
assert(same([...operations.get("in").pattern.variables.keys()], ["value", "collection"]));
assert(operations.get("in").pattern.variables.get("collection").schema === store.registered("Expressions.Basic.Collection"));
assert(same(operations.get("eq").result.schemas, [store.registered("Schemas.OfBool")])); // a result is an OfAny
assert(operations.get("get").result.schemas.length === 0 && operations.get("not").pattern.variables.get("a").schema === store.registered("Schemas.OfBool"));
const reread: any = new Builders.Store(); // the order survives a snapshot
JSON.FromJSON(reread).Snapshot(JSON.ToJSON(store).Snapshot());
assert(same([...reread.registered("Expressions.Basic.shl").pattern.variables.keys()], ["value", "count"]));
assert(same(E.DIALECT.validate(term(E.operation("contains", 1n, 2n))), ["'contains' is not an operation Basic defines"]));
assert(same(E.DIALECT.validate(term(E.operation("add", 1n))), ["add takes 2 arguments, got 1"]));
assert(!E.OPERATIONS.has("convert") && !E.OPERATIONS.has("pack")); // deferred with type inference
// Its domains: owned by name, each related to its schema, an OfAny of its scalar, both ways.
const domains = new Map<string, any>([...basic.domains].map((entry: any) => [`${entry.name}`, entry.domain]));
assert(same([...domains.keys()], [...Basic.Domains.DOMAINS.keys(), "state", "mode"]) && domains.get("uint8") === defined.get("uint8")); // EXP-06's too
const uint8 = store.registered("Expressions.Basic.uint8");
const wrap8 = new Basic.Domains.OfInteger.Builder().width(8n).signed(false).overflow("wrap").create();
assert(same(uint8.schemas, [store.registered("Schemas.OfInt")]) && same([...domains.get("uint8").schema].map((e: any) => e.schema), [uint8]));
const readThrough = [...new Builders.OfAny.Builder(uint8, false, store).domain].map((e: any) => Basic.Domains.read(e.domain));
assert(readThrough.length === 1 && readThrough[0]!.equals(wrap8)); // its schema's
assert(Basic.Domains.read(domains.get("uint8")).equals(wrap8));
assert([...domains.values()].every((d) => Basic.Domains.read(d).validate().length === 0));
assert(JSON.ToJSON(store)(DomainSchema, domains.get("uint8")).split('"wrap"').length === 2); // a document of its own, by its path
